# 📈 Stock Price Prediction — LSTM Exploration Notebook

This notebook walks through the complete ML pipeline:

1. Data Acquisition (Yahoo Finance)
2. Exploratory Data Analysis
3. Feature Engineering (Technical Indicators)
4. Preprocessing & Normalization
5. Time-Series Windowing
6. LSTM Model Training
7. Evaluation
8. Future Forecasting

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────
# !pip install yfinance tensorflow scikit-learn pandas matplotlib seaborn joblib

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'model'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#334155',
    'grid.color':       '#334155',
    'text.color':       '#e2e8f0',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
})

print('✅ All imports successful')

## 1. 📥 Data Acquisition

In [ ]:
from data_pipeline import fetch_stock_data, add_technical_indicators

# ── Configuration ─────────────────────────────
TICKER     = 'AAPL'          # Change to any ticker
START_DATE = '2015-01-01'
WINDOW     = 60
HORIZON    = 1

df_raw = fetch_stock_data(TICKER, start=START_DATE, save_dir='../data/raw')
print(f'Raw data shape: {df_raw.shape}')
df_raw.tail()

## 2. 📊 Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [3,1,1]})
fig.suptitle(f'{TICKER} — Raw OHLCV Overview', fontsize=15, fontweight='bold', color='white')

# Price
axes[0].plot(df_raw.index, df_raw['Close'], color='#3b82f6', linewidth=1.5, label='Close')
axes[0].fill_between(df_raw.index, df_raw['Low'], df_raw['High'], alpha=0.15, color='#3b82f6')
axes[0].set_ylabel('Price (USD)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df_raw.index, df_raw['Volume']/1e6, color='#0ea5e9', alpha=0.7, width=1)
axes[1].set_ylabel('Volume (M)'); axes[1].grid(True, alpha=0.3)

# Daily returns
returns = df_raw['Close'].pct_change().dropna() * 100
axes[2].bar(returns.index, returns, color=np.where(returns >= 0, '#22c55e', '#ef4444'), alpha=0.7, width=1)
axes[2].set_ylabel('Daily Return (%)'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('\n📊 Basic Statistics:')
print(df_raw['Close'].describe())

## 3. 🔧 Feature Engineering — Technical Indicators

In [ ]:
df = add_technical_indicators(df_raw)
print(f'Feature-engineered shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.tail(3)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
fig.suptitle(f'{TICKER} — Technical Indicators', fontsize=14, fontweight='bold', color='white')

recent = df.iloc[-365:]

# Price + Moving Averages + Bollinger Bands
axes[0].plot(recent.index, recent['Close'],    color='#3b82f6', lw=1.5, label='Close')
axes[0].plot(recent.index, recent['SMA_20'],   color='#f97316', lw=1.5, label='SMA-20')
axes[0].plot(recent.index, recent['SMA_50'],   color='#a855f7', lw=1.5, label='SMA-50')
axes[0].plot(recent.index, recent['EMA_20'],   color='#22c55e', lw=1.0, label='EMA-20', linestyle='--')
axes[0].fill_between(recent.index, recent['BB_Lower'], recent['BB_Upper'], alpha=0.1, color='#94a3b8', label='BB')
axes[0].set_ylabel('Price'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(recent.index, recent['Volume']/1e6, color='#0ea5e9', alpha=0.6, width=1, label='Volume')
axes[1].set_ylabel('Volume (M)'); axes[1].grid(True, alpha=0.3)

# RSI
axes[2].plot(recent.index, recent['RSI'], color='#a855f7', lw=1.5, label='RSI-14')
axes[2].axhline(70, color='#ef4444', linestyle='--', alpha=0.7, label='Overbought (70)')
axes[2].axhline(30, color='#22c55e', linestyle='--', alpha=0.7, label='Oversold (30)')
axes[2].fill_between(recent.index, 30, 70, alpha=0.05, color='white')
axes[2].set_ylim(0,100); axes[2].set_ylabel('RSI'); axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

# MACD
macd_hist = recent['MACD'] - recent['MACD_Signal']
axes[3].plot(recent.index, recent['MACD'],        color='#3b82f6', lw=1.5, label='MACD')
axes[3].plot(recent.index, recent['MACD_Signal'], color='#f97316', lw=1.5, label='Signal')
axes[3].bar(recent.index, macd_hist, color=np.where(macd_hist >= 0, '#22c55e', '#ef4444'), alpha=0.5, width=1)
axes[3].axhline(0, color='white', lw=0.5)
axes[3].set_ylabel('MACD'); axes[3].legend(fontsize=8); axes[3].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 4. 🔄 Preprocessing & Normalization

In [ ]:
from data_pipeline import normalize_data, FEATURE_COLS, CLOSE_COL_IDX
import os

os.makedirs('../data/processed', exist_ok=True)
scaler_path = f'../data/processed/{TICKER}_scaler.pkl'

scaled, scaler = normalize_data(df, FEATURE_COLS, scaler_path)

print(f'Scaled data shape: {scaled.shape}')
print(f'Value range: [{scaled.min():.4f}, {scaled.max():.4f}]')
print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

# Visualise distribution
fig, ax = plt.subplots(figsize=(14,4))
ax.boxplot(scaled[:, :6], labels=FEATURE_COLS[:6], patch_artist=True,
           boxprops=dict(facecolor='#1e3a8a', color='#3b82f6'),
           medianprops=dict(color='#f97316'))
ax.set_title('Scaled Feature Distributions (first 6 features)')
plt.tight_layout(); plt.show()

## 5. 🪟 Time-Series Window Creation

In [ ]:
from data_pipeline import create_sequences, split_data

X, y = create_sequences(scaled, window_size=WINDOW,
                        target_col_idx=CLOSE_COL_IDX,
                        forecast_horizon=HORIZON)

X_train, X_test, y_train, y_test = split_data(X, y, split_ratio=0.80)

print(f'X_train: {X_train.shape}   (samples × window × features)')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test:  {y_test.shape}')

# Visualise one sample window
fig, ax = plt.subplots(figsize=(12, 4))
sample_window = X_train[100, :, CLOSE_COL_IDX]  # Close column
ax.plot(sample_window, color='#3b82f6', lw=2, marker='o', markersize=3)
ax.axvline(WINDOW-1, color='#f97316', linestyle='--', label=f'Predict → day {WINDOW+1}')
ax.scatter([WINDOW], [y_train[100, 0]], color='#22c55e', zorder=5, s=80, label='Target')
ax.set_title(f'Sample Window (normalized Close) — Window size = {WINDOW}')
ax.set_xlabel('Time Step'); ax.set_ylabel('Scaled Price'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. 🧠 LSTM Model Architecture

In [ ]:
from lstm_model import build_lstm_model, get_callbacks
import tensorflow as tf

print(f'TensorFlow version: {tf.__version__}')

model = build_lstm_model(
    window_size      = WINDOW,
    n_features       = X_train.shape[2],
    forecast_horizon = HORIZON,
    lstm_units       = (128, 64),
    dropout_rate     = 0.2,
    learning_rate    = 1e-3,
    use_attention    = True,
)

# Count parameters
total_params = model.count_params()
print(f'\n🔢 Total parameters: {total_params:,}')

## 7. 🏋️ Model Training

In [ ]:
import os
os.makedirs('../backend/models', exist_ok=True)

callbacks = get_callbacks(
    checkpoint_path = f'../backend/models/{TICKER}/checkpoint.keras',
    patience = 15
)

history = model.fit(
    X_train, y_train,
    validation_data = (X_test, y_test),
    epochs     = 100,
    batch_size = 32,
    callbacks  = callbacks,
    shuffle    = False,
    verbose    = 1
)

print('✅ Training complete!')

In [ ]:
# ── Plot training history ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{TICKER} — Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'],     color='#3b82f6', lw=2, label='Train Loss')
axes[0].plot(history.history['val_loss'], color='#f97316', lw=2, label='Val Loss')
axes[0].set_title('Huber Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'],     color='#22c55e', lw=2, label='Train MAE')
axes[1].plot(history.history['val_mae'], color='#a855f7', lw=2, label='Val MAE')
axes[1].set_title('Mean Absolute Error'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 8. 📊 Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from data_pipeline import inverse_transform_close

y_pred_sc = model.predict(X_test, verbose=0)
n_features = X_test.shape[2]

y_pred = inverse_transform_close(scaler, y_pred_sc, CLOSE_COL_IDX, n_features)
y_true = inverse_transform_close(scaler, y_test,    CLOSE_COL_IDX, n_features)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-10))) * 100
r2   = r2_score(y_true, y_pred)
da   = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100

print(f'╔══════════════════════════════════════╗')
print(f'║        EVALUATION METRICS            ║')
print(f'╠══════════════════════════════════════╣')
print(f'║  RMSE  : ${rmse:>10.4f}               ║')
print(f'║  MAE   : ${mae:>10.4f}               ║')
print(f'║  MAPE  :  {mape:>10.2f}%             ║')
print(f'║  R²    :  {r2:>10.4f}               ║')
print(f'║  Dir.  :  {da:>10.1f}%  accuracy    ║')
print(f'╚══════════════════════════════════════╝')

In [ ]:
# ── Actual vs Predicted Plot ──────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle(f'{TICKER} — Actual vs Predicted (Test Set)', fontsize=14, fontweight='bold')

axes[0].plot(y_true, color='#3b82f6', lw=1.5, label='Actual Price')
axes[0].plot(y_pred, color='#f97316', lw=1.5, alpha=0.85, label='Predicted Price')
axes[0].fill_between(range(len(y_pred)), y_pred*0.97, y_pred*1.03,
                     color='#f97316', alpha=0.1, label='±3% confidence')
axes[0].set_ylabel('Price (USD)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

residuals = y_true - y_pred
axes[1].bar(range(len(residuals)), residuals,
            color=np.where(residuals >= 0, '#22c55e', '#ef4444'), alpha=0.7, width=0.8)
axes[1].axhline(0, color='white', lw=0.8)
axes[1].set_xlabel('Test Sample Index'); axes[1].set_ylabel('Residual (USD)')
axes[1].set_title('Residuals'); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 9. 🔮 Future Forecast

In [ ]:
from lstm_model import save_model
import json

os.makedirs(f'../backend/models/{TICKER}', exist_ok=True)
model_path = f'../backend/models/{TICKER}/model.keras'
save_model(model, model_path)

config = {
    'ticker':           TICKER,
    'window_size':      WINDOW,
    'forecast_horizon': HORIZON,
    'n_features':       n_features,
    'feature_cols':     FEATURE_COLS,
    'metrics':          {'rmse': round(rmse,4), 'mae': round(mae,4),
                         'mape': round(mape,4), 'r2': round(r2,4),
                         'directional_accuracy': round(da/100, 4)},
    'model_path':       model_path,
    'scaler_path':      scaler_path,
}
with open(f'../backend/models/{TICKER}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('✅ Model + config saved!')

In [ ]:
# ── Generate future predictions ───────────────────────────────────────
import pandas as pd

N_FUTURE = 30
sequence = scaled[-WINDOW:].copy()
future_prices = []

for _ in range(N_FUTURE):
    x = sequence[-WINDOW:].reshape(1, WINDOW, n_features)
    pred_sc = model.predict(x, verbose=0)[0, 0]
    next_row = sequence[-1].copy()
    next_row[CLOSE_COL_IDX] = pred_sc
    sequence = np.vstack([sequence, next_row])
    price = inverse_transform_close(scaler, np.array([[pred_sc]]), CLOSE_COL_IDX, n_features)[0]
    future_prices.append(price)

last_date    = df.index[-1]
future_dates = pd.bdate_range(start=last_date, periods=N_FUTURE + 1)[1:]

# ── Plot ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))

# Historical
hist_days = 90
ax.plot(df.index[-hist_days:], df['Close'].iloc[-hist_days:],
        color='#3b82f6', lw=2, label='Historical')

# Forecast
ax.plot(future_dates, future_prices,
        color='#f97316', lw=2.5, linestyle='--', label=f'{N_FUTURE}-Day Forecast')
ax.fill_between(future_dates,
                [p * 0.97 for p in future_prices],
                [p * 1.03 for p in future_prices],
                alpha=0.15, color='#f97316', label='±3% Confidence Band')

ax.axvline(df.index[-1], color='#64748b', linestyle=':', lw=1)
ax.set_title(f'{TICKER} — {N_FUTURE}-Day Future Price Forecast',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Table
forecast_df = pd.DataFrame({'Date': future_dates, 'Predicted Price': future_prices})
forecast_df['Change'] = forecast_df['Predicted Price'].diff().round(4)
forecast_df['% Change'] = (forecast_df['Predicted Price'].pct_change() * 100).round(4)
print(f'\n📅 {N_FUTURE}-Day Forecast ({TICKER}):')
print(forecast_df.to_string(index=False))

## ✅ Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Data Acquisition (Yahoo Finance) | ✅ |
| 2 | Exploratory Data Analysis | ✅ |
| 3 | Feature Engineering (16 features) | ✅ |
| 4 | Min-Max Normalization | ✅ |
| 5 | Sliding Window (60d look-back) | ✅ |
| 6 | LSTM + Attention Architecture | ✅ |
| 7 | Model Training with Early Stopping | ✅ |
| 8 | Evaluation (RMSE, MAE, MAPE, R²) | ✅ |
| 9 | 30-Day Future Forecast | ✅ |

---

**Next Steps:**
- Start the FastAPI backend: `cd ../backend && uvicorn main:app --reload`
- Start the React frontend: `cd ../frontend && npm run dev`